In [ ]:
source("helpers/init.r")

In [ ]:
shh(library(ggwordcloud))
shh(library(cowplot))

In [ ]:
cohorts    <- fread(paste0(READY_DATA, "cohorts.txt"))
plt_cohort <- cohorts %>% filter(!ref) %>% pull(cohort) %>% unique()
ref_cohort <- cohorts %>% filter(ref)  %>% pull(cohort) %>% unique()

# 0 - Load data

In [ ]:
markers <- runQuery("markers")
base       <- cohorts %>% left_join(markers, by = "sampleId")
base       <- base %>%
  mutate(across(where(is.character), ~ na_if(., "NULL"))) %>%   # DB may return string "NULL" for missing values
  mutate(cohort = factor(cohort, levels = c(plt_cohort, ref_cohort)))
color_map  <- c(fills[["plt"]], fills[["ref"]])
names(color_map) <- c(plt_cohort, ref_cohort)

In [ ]:
validate_columns(markers, c("sampleId", "tmbPerMb", "tmbStatus", "msStatus",
                          "msIndelsPerMb", "purity", "ploidy", "svTmb",
                          "lohGenomePct", "hrStatus"), "markers")

# Helper — violin panel factory

In [ ]:
make_violin_panel <- function(data, var_labels, log_vars = character(0),
                               panel_title = "", tool_label = "",
                               thresholds = list(), ncol = NULL,
                               fixed_y = FALSE, quartile_lines = FALSE,
                               median_labels = TRUE, pct_vars = character(0)) {
  long <-
    data %>%
    select(sampleId, cohort, all_of(names(var_labels))) %>%
    pivot_longer(-c(sampleId, cohort), names_to = "var", values_to = "value") %>%
    mutate(
      label   = factor(var_labels[var], levels = unname(var_labels)),
      log_var = var %in% log_vars
    ) %>%
    filter(!is.na(value))

  # Per-facet Wilcoxon significance
  sig_df <- bind_rows(lapply(names(var_labels), function(v) {
    tmp <- long %>% filter(var == v)
    if (n_distinct(tmp$cohort) < 2 || nrow(tmp) < 4) return(NULL)
    p <- tryCatch(
      wilcox.test(value ~ cohort, data = tmp, exact = FALSE)$p.value,
      error = function(e) NA_real_
    )
    lbl <- case_when(
      is.na(p)  ~ "",
      p < 0.001 ~ "***",
      p < 0.01  ~ "**",
      p < 0.05  ~ "*",
      TRUE      ~ "ns"
    )
    data.frame(label = factor(var_labels[[v]], levels = unname(var_labels)),
               sig = lbl, stringsAsFactors = FALSE)
  }))

  # Median label data
  med_df <- if (median_labels) {
    long %>%
      group_by(label, cohort, var) %>%
      summarise(med = median(value, na.rm = TRUE), .groups = "drop") %>%
      mutate(
        txt   = ifelse(var %in% pct_vars,
                       paste0(round(med * 100, 0), "%"),
                       ifelse(med < 10,
                              as.character(round(med, 1)),
                              as.character(round(med, 0)))),
        var   = var,
        hjust = 0.5
      )
  } else NULL

  # Quartile label data
  quant_df <- if (quartile_lines) {
    long %>%
      group_by(label, cohort) %>%
      summarise(
        Q1  = quantile(value, 0.25, na.rm = TRUE),
        Med = median(value,          na.rm = TRUE),
        Q3  = quantile(value, 0.75, na.rm = TRUE),
        .groups = "drop"
      ) %>%
      pivot_longer(c(Q1, Med, Q3), names_to = "stat", values_to = "value") %>%
      mutate(
        txt   = ifelse(value < 10,
                       as.character(round(value, 1)),
                       as.character(round(value, 0))),
        hjust = ifelse(cohort == cohort_levels[1], 1.25, -0.25)
      )
  } else NULL

  p <-
    ggplot(long, aes(x = cohort, y = value, fill = cohort)) +
    geom_violin(alpha = 0.7, color = "black", linewidth = 0.3, scale = "width") +
    geom_jitter(aes(color = cohort), width = 0.15, alpha = 0.15,
                size = 0.5, show.legend = FALSE) +
    { if (!quartile_lines)
        stat_summary(
          fun = median, fun.min = median, fun.max = median,
          geom = "errorbar", width = 0.85,
          color = "black", linewidth = 0.55
        )
    } +
    { if (quartile_lines)
        stat_summary(
          fun.data = function(x) {
            qs <- quantile(x, c(0.25, 0.5, 0.75), na.rm = TRUE)
            data.frame(ymin = qs[1], y = qs[2], ymax = qs[3])
          },
          geom = "crossbar", width = 0.35, fill = NA,
          color = "black", linewidth = 0.35, fatten = 2
        )
    } +
    { if (median_labels && !is.null(med_df))
        geom_text(data = med_df,
                  aes(x = cohort, y = med, label = txt),
                  hjust = 0.5, vjust = -0.3,
                  size = 3.0, color = "#333333", fontface = "bold",
                  inherit.aes = FALSE)
    } +
    { if (quartile_lines && !is.null(quant_df))
        geom_text(data = quant_df,
                  aes(x = cohort, y = value, label = txt, hjust = hjust),
                  size = 2.3, color = "#333333", inherit.aes = FALSE)
    } +
    scale_fill_manual(values  = color_map) +
    scale_color_manual(values = color_map) +
    scale_x_discrete(guide = guide_axis(n.dodge = 2)) +
    { if (length(log_vars) > 0)
        scale_y_continuous(
          trans  = "log1p",
          breaks = c(0, 1, 10, 100, 1000, 10000),
          labels = scales::label_number(big.mark = ",")
        )
      else
        scale_y_continuous() } +
    facet_wrap(~ label, scales = if (fixed_y) "fixed" else "free_y",
               nrow = if (is.null(ncol)) 1 else NULL, ncol = ncol) +
    { if (nrow(sig_df) > 0)
        geom_text(data = sig_df, aes(x = 1.5, y = Inf, label = sig),
                  vjust = 1.3, size = 3.5, color = "#555555", inherit.aes = FALSE) } +
    labs(title = panel_title,
         subtitle = if (nchar(tool_label) > 0) paste0(tool_icon(), " ", tool_label) else "") +
    base_theme + no_y_title + no_ticks + no_x_label + no_legend +
    theme(panel.grid = element_blank()) +
    theme(
      strip.text    = element_text(size = 8, face = "bold"),
      axis.title    = element_blank(),
      axis.text.x   = element_blank(),
      axis.ticks.x  = element_blank(),
      plot.subtitle = element_markdown(hjust = 0, size = 11, color = "#304b99", face = "bold", margin = margin(b = 4)),
      plot.title    = element_text(size = 11, face = "bold", color = "#333333")
    )

  if (length(thresholds) > 0) {
    thr_df <- data.frame(
      var        = names(thresholds),
      yintercept = unlist(thresholds),
      stringsAsFactors = FALSE
    ) %>%
    filter(var %in% names(var_labels)) %>%
    mutate(label = factor(var_labels[var], levels = unname(var_labels)))
    p <- p + geom_hline(data = thr_df, aes(yintercept = yintercept),
                        color = "red", linetype = "dashed",
                        linewidth = 0.4, inherit.aes = FALSE)
  }

  p
}


# 1 - WGS tumor characteristics

In [ ]:
tc_names <- unlist(bar_names)

tcs <-
  base %>%
  drop_na(tmbStatus) %>%
  select(sampleId, cohort, all_of(names(bar_names))) %>%
  mutate(
    cdr3Total      = as.integer(cdr3Total > CDR3_INFILTRATED_THRESHOLD),
    gieHlaLoh      = as.integer(gieHlaLoh > 0),
    hrStatus       = as.integer(hrStatus == "HR_DEFICIENT"),
    tmbStatus      = as.integer(tmbStatus == "HIGH"),
    msStatus       = as.integer(msStatus == "MSI"),
    svDoubleMinute = as.integer(svDoubleMinute > 0),
    svTmb          = as.integer(svTmb > SV_HIGH_THRESHOLD),
    viralReported  = as.integer(viralReported > 0)
  ) %>%
  pivot_longer(-c(sampleId, cohort), names_to = "feature", values_to = "val") %>%
  group_by(cohort, feature) %>%
  summarise(val = mean(val, na.rm = TRUE), .groups = "drop") %>%
  rowwise() %>%
  mutate(name = factor(namer(feature, bar_names), levels = tc_names)) %>%
  ungroup() %>%
  mutate(cohort = factor(cohort, levels = c(plt_cohort, ref_cohort)))


In [ ]:
# Background highlight for immunotherapy/immune group
immune_xmin <- which(tc_names == "TMB High") - 0.5
immune_xmax <- length(tc_names) + 0.5

tumor_char_plot <-
  ggplot(tcs, aes(x = name, y = val, fill = cohort)) +
  annotate("text", x = (immune_xmin + immune_xmax) / 2, y = 0.73,
           label = "Immunotherapy / Immune", size = 2.8,
           color = "#aaaaaa", fontface = "italic", hjust = 0.5, vjust = 1) +
  annotate("segment", x = immune_xmin, xend = immune_xmax, y = 0.695, yend = 0.695,
           color = "#aaaaaa", linewidth = 0.4, alpha = 0.5) +
  geom_bar(stat = "identity", width = .7, position = "dodge",
           linewidth = 0.6, color = "black", alpha = 0.7) +
  scale_fill_manual(values = color_map) +
  scale_x_discrete(guide = guide_axis(n.dodge = 2)) +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                     breaks = c(0, 1),
                     expand = expansion(mult = c(0, .18))) +
  geom_text(aes(label = scales::percent(round(val, 2), accuracy = 1)),
            position = position_dodge(width = .7),
            color = "black", size = 3.3, vjust = -.4) +
  labs(subtitle = paste0(tool_icon(), " PURPLE · LINX · SAGE · CHORD · LILAC · VirusInterpreter")) +
  base_theme +
  theme(
    axis.title        = element_blank(),
    axis.text.x       = element_text(size = 9),
    plot.subtitle     = element_markdown(hjust = 0, size = 11, color = "#304b99", face = "bold", margin = margin(b = 4)),
    legend.position   = c(1, 1),
    legend.justification = c(1, 1),
    legend.direction  = "horizontal",
    legend.title      = element_blank(),
    legend.text       = element_text(size = 13),
  )

# 2 - Genome structure

In [ ]:
purity_plot <- make_violin_panel(
  base,
  var_labels  = c(purity = "Tumor purity"),
  pct_vars    = "purity",
  thresholds  = list(purity = PURITY_LOW_THRESHOLD),
  panel_title = "Genome Structure",
  tool_label  = "PURPLE"
) +
  geom_rect(
    data        = data.frame(ymin = 0, ymax = PURITY_LOW_THRESHOLD),
    aes(xmin = -Inf, xmax = Inf, ymin = ymin, ymax = ymax),
    fill        = "#fcaf17",
    alpha       = 0.18,
    inherit.aes = FALSE
  ) +
  annotate("text",
    x        = Inf,
    y        = 0,
    label    = "Low purity",
    hjust    = 1.05,
    vjust    = -0.3,
    size     = 2.2,
    color    = "#b07800",
    fontface = "italic"
  ) +
  scale_y_continuous(
    breaks = c(0.2, 0.5, 1.0),
    labels = scales::percent_format(accuracy = 1),
    limits = c(0, 1.05)
  )

ploidy_plot <- make_violin_panel(
  base,
  var_labels  = c(ploidy = "Ploidy"),
  panel_title = "",
  tool_label  = ""
)

genome_struct_plot <- plot_grid(purity_plot, ploidy_plot, nrow = 1, rel_widths = c(1, 1))


# 2 - Mutational burden

In [ ]:
tmb_shade_df <- data.frame(
  label = factor(
    c("TMB (mut/Mb)", "MSI (indels/Mb)"),
    levels = c("TMB (mut/Mb)", "MSI (indels/Mb)", "SNVs (per Mb)", "INDELs (per Mb)", "MNVs (per Mb)")
  ),
  ymin = c(TMB_HIGH_THRESHOLD, MSI_HIGH_THRESHOLD),
  txt  = c("TMB High", "MSI High")
)

mut_burden_plot <- make_violin_panel(
  base,
  var_labels  = c(tmbPerMb      = "TMB (mut/Mb)",
                  msIndelsPerMb = "MSI (indels/Mb)",
                  snpPerMb      = "SNVs (per Mb)",
                  indelsPerMb   = "INDELs (per Mb)",
                  mnpPerMb      = "MNVs (per Mb)"),
  log_vars    = c("tmbPerMb", "msIndelsPerMb", "snpPerMb", "indelsPerMb", "mnpPerMb"),
  thresholds  = list(tmbPerMb = TMB_HIGH_THRESHOLD, msIndelsPerMb = MSI_HIGH_THRESHOLD),
  fixed_y     = TRUE,
  panel_title = "Mutational burden",
  tool_label  = "SAGE · PAVE · PURPLE"
) +
  geom_rect(
    data        = tmb_shade_df,
    aes(xmin = -Inf, xmax = Inf, ymin = ymin, ymax = Inf),
    fill        = "red",
    alpha       = 0.08,
    inherit.aes = FALSE
  ) +
  geom_text(
    data        = tmb_shade_df,
    aes(x = Inf, y = Inf, label = txt),
    hjust       = 1.05,
    vjust       = 1.4,
    size        = 2.2,
    color       = "#cc3333",
    fontface    = "italic",
    inherit.aes = FALSE
  ) +
  scale_y_continuous(
    trans   = "log1p",
    breaks  = c(0, 1, 10, 100, 1000, 10000),
    labels  = function(x) ifelse(x == 0, "", scales::number(x, big.mark = ","))
  )


# 2 - Genome integrity

In [ ]:
telomere_plot <- make_violin_panel(
  base,
  var_labels  = c(somaticGermlineTelomereRatio = "Somatic/germline
telomere ratio"),
  log_vars    = c("somaticGermlineTelomereRatio"),
  panel_title = "Telomere length",
  tool_label  = "TEAL"
) +
  scale_y_continuous(
    trans   = "log1p",
    limits  = c(0, NA),
    breaks  = c(0, 1, 10),
    labels  = scales::label_number(big.mark = ",")
  )


# 3 - CUPPA

In [ ]:
cuppa_freq <-
  base %>%
  filter(!is.na(cuppaCancertype_DNA_COMBINED),
         cuppaProb_DNA_COMBINED >= CUPPA_DNA_COMBINED_THRESHOLD,
         cohort == plt_cohort) %>%
  count(cuppaCancertype_DNA_COMBINED) %>%
  mutate(
    pct   = n / sum(n),
    label = ifelse(cuppaCancertype_DNA_COMBINED %in% names(cuppa_name_map),
                   cuppa_name_map[cuppaCancertype_DNA_COMBINED],
                   cuppaCancertype_DNA_COMBINED)
  )

cuppa_n_shown <- sum(cuppa_freq$n)

cuppa_plot <-
  ggplot(cuppa_freq, aes(label = label, size = pct, x = 0, y = 0)) +
  geom_text_wordcloud(seed = 42, show.legend = FALSE, eccentricity = 1,
                      rm_outside = TRUE, color = fills[["plt"]], area_corr = FALSE) +
  scale_size_area(max_size = 16) +
  scale_x_continuous(limits = c(-1, 1), expand = expansion(0)) +
  scale_y_continuous(limits = c(-1, 1), expand = expansion(0)) +
  labs(title = "Primary tumor prediction",
       subtitle = paste0(tool_icon(), " CUPPA  ·  n=", cuppa_n_shown,
                         "  (prob ≥ ", CUPPA_DNA_COMBINED_THRESHOLD, ")")) +
  base_theme +
  theme(
    plot.title    = element_text(size = 11, face = "bold", color = "#333333"),
    plot.subtitle = element_markdown(hjust = 0, size = 11, color = "#304b99", face = "bold", margin = margin(b = 4)),
    plot.caption  = element_text(size = 7, color = "#aaaaaa", hjust = 0, face = "italic"),
    plot.margin   = margin(t = 5.5, r = 5.5, b = 5.5, l = 10),
    axis.text     = element_blank(),
    axis.ticks    = element_blank(),
    axis.title    = element_blank()
  )

In [ ]:
# 3b - CUPPA extended (all classifiers)

In [ ]:
.cuppa_panel <- function(type_col, prob_col, panel_title,
                         threshold = CUPPA_PROB_THRESHOLD,
                         best_estimate = FALSE,
                         prob_label = NULL,
                         max_size = 14) {
  freq <- base %>%
    filter(!is.na(.data[[type_col]]),
           if (best_estimate) TRUE else .data[[prob_col]] >= threshold,
           cohort == plt_cohort) %>%
    count(.data[[type_col]]) %>%
    rename(cancer_type = 1) %>%
    mutate(
      pct   = n / sum(n),
      label = ifelse(cancer_type %in% names(cuppa_name_map),
                     cuppa_name_map[cancer_type], cancer_type)
    )

  n_shown  <- sum(freq$n)
  subtitle <- if (!is.null(prob_label)) paste0("n=", n_shown, "  (", prob_label, ")")
              else if (best_estimate)   paste0("n=", n_shown, "  (Best location estimate)")
              else                      paste0("n=", n_shown, "  (prob ≥ ", threshold, ")")

  ggplot(freq, aes(label = label, size = pct, x = 0, y = 0)) +
    geom_text_wordcloud(seed = 42, show.legend = FALSE, eccentricity = 1,
                        rm_outside = TRUE, color = fills[["plt"]],
                        area_corr = FALSE) +
    scale_size_area(max_size = max_size) +
    scale_x_continuous(limits = c(-1, 1), expand = expansion(0)) +
    scale_y_continuous(limits = c(-1, 1), expand = expansion(0)) +
    labs(title = panel_title, subtitle = subtitle) +
    base_theme +
    theme(
      axis.text    = element_blank(),
      axis.ticks   = element_blank(),
      axis.title   = element_blank(),
      plot.title    = element_text(size = 10, face = "bold", color = "#333333", hjust = 0.5),
      plot.subtitle = element_text(size = 8, color = "grey50", hjust = 0.5),
      plot.margin   = margin(5.5, 5.5, 5.5, 5.5)
    )
}

cuppa_extended_plot <- plot_grid(
  .cuppa_panel("cuppaCancertype_DNA_COMBINED", "cuppaProb_DNA_COMBINED", "DNA Combined",
               threshold = CUPPA_DNA_COMBINED_THRESHOLD, prob_label = "probability > 80%",
               max_size = 14),
  .cuppa_panel("cuppaCancertype_GEN_POS",      "cuppaProb_GEN_POS",      "Genomic Position",
               best_estimate = TRUE, max_size = 9),
  .cuppa_panel("cuppaCancertype_EVENT",          "cuppaProb_EVENT",          "Genomic Events",
               best_estimate = TRUE, max_size = 9),
  .cuppa_panel("cuppaCancertype_SNV96",         "cuppaProb_SNV96",         "SBS96",
               best_estimate = TRUE, max_size = 9),
  nrow = 1, ncol = 4
)


In [ ]:
.cuppa_bar_panel <- function(type_col, prob_col, panel_title,
                            threshold = CUPPA_PROB_THRESHOLD,
                            best_estimate = FALSE,
                            prob_label = NULL) {
  freq <- base %>%
    filter(!is.na(.data[[type_col]]),
           if (best_estimate) TRUE else .data[[prob_col]] >= threshold,
           cohort == plt_cohort) %>%
    count(.data[[type_col]]) %>%
    rename(cancer_type = 1) %>%
    mutate(
      pct   = n / sum(n),
      label = ifelse(cancer_type %in% names(cuppa_name_map),
                     cuppa_name_map[cancer_type], cancer_type)
    ) %>%
    arrange(pct) %>%
    mutate(label = factor(label, levels = label))

  n_shown  <- sum(freq$n)
  subtitle <- if (!is.null(prob_label)) paste0("n=", n_shown, "  (", prob_label, ")")
              else if (best_estimate)   paste0("n=", n_shown, "  (Best location estimate)")
              else                      paste0("n=", n_shown, "  (prob \u2265 ", threshold, ")")

  ggplot(freq, aes(x = pct, y = label)) +
    geom_bar(stat = "identity", fill = fills[["plt"]], alpha = 0.75,
             color = "black", linewidth = 0.3, width = 0.7) +
    geom_text(aes(label = scales::percent(pct, accuracy = 1)),
              hjust = -0.15, size = 3, color = "#333333") +
    scale_x_continuous(expand = expansion(mult = c(0, 0.28)),
                       labels = scales::percent_format(accuracy = 1)) +
    labs(title = panel_title, subtitle = subtitle) +
    base_theme +
    theme(
      axis.title        = element_blank(),
      axis.text.x       = element_blank(),
      axis.ticks        = element_blank(),
      plot.title        = element_text(size = 10, face = "bold", color = "#333333", hjust = 0.5),
      plot.subtitle     = element_text(size = 8, color = "grey50", hjust = 0.5),
      plot.margin       = margin(5.5, 5.5, 5.5, 5.5),
      panel.grid.major.x = element_blank()
    )
}

cuppa_bar_plot <- plot_grid(
  .cuppa_bar_panel("cuppaCancertype_DNA_COMBINED", "cuppaProb_DNA_COMBINED", "DNA Combined",
                   threshold = CUPPA_DNA_COMBINED_THRESHOLD, prob_label = "probability > 80%"),
  .cuppa_bar_panel("cuppaCancertype_GEN_POS",      "cuppaProb_GEN_POS",      "Genomic Position",
                   best_estimate = TRUE),
  .cuppa_bar_panel("cuppaCancertype_SNV96",         "cuppaProb_SNV96",         "SBS96",
                   best_estimate = TRUE),
  .cuppa_bar_panel("cuppaCancertype_EVENT",          "cuppaProb_EVENT",          "Genomic Events",
                   best_estimate = TRUE),
  nrow = 1, ncol = 4
)


In [ ]:
# ── CUPPA stacked bar plot (9c) — top 5 cancer types + Low Conf ─────────
# Thresholds: DNA Combined >= 0.8, others >= 0.5; below threshold → Low Conf
cuppa_classifiers <- list(
  list(type  = "cuppaCancertype_DNA_COMBINED", prob = "cuppaProb_DNA_COMBINED",
       label = "DNA Combined",     threshold = CUPPA_DNA_COMBINED_THRESHOLD),
  list(type  = "cuppaCancertype_GEN_POS",      prob = "cuppaProb_GEN_POS",
       label = "Genomic Position", threshold = CUPPA_PROB_THRESHOLD),
  list(type  = "cuppaCancertype_SNV96",         prob = "cuppaProb_SNV96",
       label = "SBS96",            threshold = CUPPA_PROB_THRESHOLD),
  list(type  = "cuppaCancertype_EVENT",          prob = "cuppaProb_EVENT",
       label = "Genomic Events",   threshold = CUPPA_PROB_THRESHOLD)
)

# Build long data: assign Low Conf for samples below threshold
raw_stacked <- bind_rows(lapply(cuppa_classifiers, function(cl) {
  df <- base %>%
    filter(!is.na(.data[[cl$type]]), cohort == plt_cohort) %>%
    mutate(
      cancer_type = .data[[cl$type]],
      prob        = suppressWarnings(as.numeric(.data[[cl$prob]])),
      raw_label   = ifelse(cancer_type %in% names(cuppa_name_map),
                           cuppa_name_map[cancer_type], cancer_type),
      label       = ifelse(is.na(prob) | prob < cl$threshold, "Low Conf", raw_label),
      classifier  = cl$label,
      n_total     = n()
    )
  df %>% select(classifier, label, n_total)
}))

# Top 4 types from high-confidence entries across all classifiers
top4_types <- raw_stacked %>%
  filter(label != "Low Conf") %>%
  count(label) %>%
  arrange(desc(n)) %>%
  slice_head(n = 4) %>%
  pull(label)

stacked_data <- raw_stacked %>%
  mutate(
    label = case_when(
      label == "Low Conf" ~ "Low Conf",
      label %in% top4_types    ~ label,
      TRUE                     ~ "Other"
    )
  ) %>%
  group_by(classifier, label) %>%
  mutate(n_total = first(n_total)) %>%
  count(classifier, label, n_total) %>%
  mutate(pct = n / n_total)

type_levels <- c(top4_types, "Other", "Low Conf")
type_colors <- setNames(
  c("#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#cccccc", "white"),
  c(top4_types, "Other", "Low Conf")
)[type_levels]

stacked_data <- stacked_data %>%
  mutate(
    label      = factor(label, levels = rev(type_levels)),
    classifier = factor(classifier,
                        levels = rev(c("DNA Combined", "Genomic Position",
                                       "Genomic Events", "SBS96")))
  )

cuppa_stacked_plot <-
  ggplot(stacked_data, aes(x = pct, y = classifier, fill = label)) +
  geom_bar(stat = "identity", color = "black", linewidth = 0.25, width = 0.65, alpha = 0.7) +
  geom_text(aes(label = ifelse(pct >= 0.05, scales::percent(pct, accuracy = 1), "")),
            position = position_stack(vjust = 0.5),
            size = 3, color = "white", fontface = "bold") +
  scale_x_continuous(labels = scales::percent_format(accuracy = 1),
                     expand = expansion(mult = c(0, 0.01))) +
  scale_fill_manual(values = type_colors, breaks = rev(type_levels)) +
  labs(title    = "Primary Tumor Prediction",
       subtitle = paste0(tool_icon(), " CUPPA"),
       fill     = NULL) +
  guides(fill = guide_legend(reverse = TRUE)) +
  base_theme + no_y_title +
  theme(
    axis.title.x  = element_blank(),
    axis.text.x   = element_blank(),
    axis.ticks    = element_blank(),
    plot.title    = element_text(size = 11, face = "bold", color = "#333333", hjust = 0.5),
    plot.subtitle = element_markdown(hjust = 0.5, size = 11, color = "#304b99",
                                     face = "bold", margin = margin(b = 4)),
    legend.position  = "right",
    legend.text      = element_text(size = 9),
    panel.grid.major.x = element_blank()
  )


# 3 - Structural variants

In [ ]:
sv_vars <- c(
  svTmb             = "SV burden",
  svDeletions       = "Deletions",
  svDuplications    = "Duplications",
  svInversions      = "Inversions",
  svTranslocations  = "Translocations",
  svComplexClusters = "Complex clusters",
  svLine            = "LINE insertions"
)

sv_log_vars <- names(sv_vars)

# Sort by descending median across all samples
sv_medians <- sapply(names(sv_vars), function(v) median(base[[v]], na.rm = TRUE))
sv_vars    <- sv_vars[order(sv_medians, decreasing = TRUE)]

sv_plot <- make_violin_panel(
  base,
  var_labels      = sv_vars,
  log_vars        = sv_log_vars,
  fixed_y         = TRUE,
  panel_title     = "Structural variants",
  tool_label      = "ESVEE · LINX",
  median_labels   = TRUE
) +
  scale_y_continuous(
    trans   = "log1p",
    breaks  = c(0, 1, 10, 100, 1000, 10000),
    labels  = function(x) ifelse(x == 0, "", scales::number(x, big.mark = ","))
  )


# 3 - Immune

In [ ]:
tcell_shade_df <- data.frame(
  label = factor("T-cell", levels = c("T-cell", "B-cell")),
  ymin  = TCELL_HIGH_THRESHOLD,
  txt   = "T-cell High"
)

immune_plot <- make_violin_panel(
  base,
  var_labels  = c(cdr3Tcell = "T-cell",
                  cdr3Bcell = "B-cell"),
  log_vars    = c("cdr3Tcell", "cdr3Bcell"),
  fixed_y     = TRUE,
  thresholds  = list(cdr3Tcell = TCELL_HIGH_THRESHOLD),
  panel_title = "Immune Infiltration",
  tool_label  = "CIDER"
) +
  geom_rect(
    data        = tcell_shade_df,
    aes(xmin = -Inf, xmax = Inf, ymin = ymin, ymax = Inf),
    fill        = "red",
    alpha       = 0.08,
    inherit.aes = FALSE
  ) +
  geom_text(
    data        = tcell_shade_df,
    aes(x = Inf, y = Inf, label = txt),
    hjust       = 1.05,
    vjust       = 1.4,
    size        = 2.2,
    color       = "#cc3333",
    fontface    = "italic",
    inherit.aes = FALSE
  ) +
  scale_y_continuous(
    trans   = "log1p",
    breaks  = c(0, 1, 10, 100, 1000, 10000),
    labels  = function(x) ifelse(x == 0, "", scales::number(x, big.mark = ","))
  )


# 4 - RNA expression

In [ ]:
rna_selected <- c(
  "rnaProliferation", "rnaIfnGamma",  "rnaTgfBeta",  "rnaEmt",
  "rnaHypoxia",       "rnaMycTargets", "rnaWnt",     "rnaP53Pathway"
)

rna_cols      <- intersect(rna_selected, colnames(base))
rna_available <- length(rna_cols) > 0 && any(!is.na(base[, ..rna_cols]))

# Sort by descending median of test cohort.
# Filter to RNA-available samples only (sentinel = 0 or NA) before computing medians,
# matching the same filter applied in the violin/density plots.
if (rna_available) {
  sentinel_col <- intersect("rnaProliferation", rna_cols)
  base_plt <- base %>%
    filter(cohort == plt_cohort)
  if (length(sentinel_col) > 0) {
    base_plt <- base_plt %>%
      filter(!is.na(as.numeric(.data[[sentinel_col]])),
             as.numeric(.data[[sentinel_col]]) != 0)
  }
  medians <- sapply(rna_cols, function(v) {
    vals <- base_plt %>% pull(all_of(v)) %>% as.numeric()
    median(vals, na.rm = TRUE)
  })
  message("RNA medians (", plt_cohort, ", n=", nrow(base_plt), "): ",
          paste(rna_cols, round(medians, 2), sep = "=", collapse = ", "))
  rna_cols <- rna_cols[order(medians, decreasing = TRUE)]
  message("RNA panel order: ", paste(rna_cols, collapse = ", "))
}

rna_plot <- if (rna_available) {
  # Back-transform log2 adjTPM -> adjTPM for interpretable axis values
  base_rna <- copy(base)
  base_rna[, (rna_cols) := lapply(.SD, as.numeric), .SDcols = rna_cols]

  # Drop samples without RNA data — rnaProliferation is NA (or 0 sentinel)
  # for samples with no RNA sequencing (same logic as 1_clinical.ipynb).
  if ("rnaProliferation" %in% rna_cols) {
    base_rna <- base_rna[!is.na(base_rna$rnaProliferation) &
                           base_rna$rnaProliferation != 0, ]
  }

  for (col in rna_cols) base_rna[, (col) := 2^get(col) - 1]

  make_violin_panel(
    base_rna,
    var_labels  = marker_names[rna_cols],
    ncol        = 8,
    fixed_y     = TRUE,
    panel_title = "",
    tool_label  = "ISOFOX"
  ) +
  theme(plot.title = element_blank()) +
  scale_y_continuous(
    trans  = "log1p",
    breaks = c(0, 1, 10, 100, 1000),
    labels = scales::label_number(big.mark = ",")
  )
} else {
  NULL
}

In [ ]:
rna_density_plot <- if (rna_available) {
  base_rna_d <- copy(base)
  base_rna_d[, (rna_cols) := lapply(.SD, as.numeric), .SDcols = rna_cols]
  if ("rnaProliferation" %in% rna_cols) {
    base_rna_d <- base_rna_d[!is.na(base_rna_d$rnaProliferation) &
                                base_rna_d$rnaProliferation != 0, ]
  }
  for (col in rna_cols) base_rna_d[, (col) := 2^get(col) - 1]

  long_d <- base_rna_d %>%
    select(sampleId, cohort, all_of(rna_cols)) %>%
    pivot_longer(-c(sampleId, cohort), names_to = "var", values_to = "value") %>%
    mutate(label = factor(marker_names[var], levels = marker_names[rna_cols])) %>%
    filter(!is.na(value))

  ggplot(long_d, aes(x = value + 1, fill = cohort, color = cohort)) +
    geom_density(alpha = 0.45, linewidth = 0.35) +
    scale_fill_manual(values  = color_map) +
    scale_color_manual(values = color_map) +
    scale_x_continuous(
      trans  = "log10",
      breaks = c(1, 10, 100, 1000),
      labels = c("0", "10", "100", "1k")
    ) +
    facet_wrap(~ label, nrow = 1, scales = "free_y") +
    labs(subtitle = paste0(tool_icon(), " ISOFOX")) +
    base_theme + no_y_title + no_ticks + no_legend +
    theme(
      strip.text    = element_text(size = 8, face = "bold"),
      axis.title.x  = element_blank(),
      axis.text.x   = element_text(size = 7),
      plot.subtitle = element_markdown(hjust = 0, size = 11, color = "#304b99",
                                       face = "bold", margin = margin(b = 4))
    )
} else NULL


In [ ]:
rna_quantile_plot <- if (rna_available) {
  base_rna_q <- copy(base)
  base_rna_q[, (rna_cols) := lapply(.SD, as.numeric), .SDcols = rna_cols]
  if ("rnaProliferation" %in% rna_cols) {
    base_rna_q <- base_rna_q[!is.na(base_rna_q$rnaProliferation) &
                                base_rna_q$rnaProliferation != 0, ]
  }
  for (col in rna_cols) base_rna_q[, (col) := 2^get(col) - 1]

  long_q <- base_rna_q %>%
    select(sampleId, cohort, all_of(rna_cols)) %>%
    pivot_longer(-c(sampleId, cohort), names_to = "var", values_to = "value") %>%
    mutate(label = factor(marker_names[var], levels = marker_names[rna_cols])) %>%
    filter(!is.na(value)) %>%
    group_by(label, cohort) %>%
    arrange(value) %>%
    mutate(quantile = (row_number() - 0.5) / n()) %>%
    ungroup()

  med_q <- long_q %>%
    group_by(label, cohort) %>%
    summarise(med = median(value, na.rm = TRUE), .groups = "drop")

  ggplot(long_q, aes(x = quantile, y = value, color = cohort)) +
    geom_line(linewidth = 0.6, alpha = 0.85) +
    geom_text(data = med_q,
              aes(x = 0.5, y = med,
                  label = ifelse(med < 10, as.character(round(med, 1)),
                                 as.character(round(med, 0))),
                  color = cohort),
              vjust = -0.5, size = 2.5, fontface = "bold",
              show.legend = FALSE, inherit.aes = FALSE) +
    scale_color_manual(values = color_map) +
    scale_y_continuous(
      trans  = "log1p",
      breaks = c(0, 1, 10, 100, 1000),
      labels = scales::label_number(big.mark = ",")
    ) +
    scale_x_continuous(
      breaks = c(0, 0.5, 1),
      labels = scales::percent_format(accuracy = 1)
    ) +
    facet_wrap(~ label, nrow = 1, scales = "fixed") +
    labs(x = "Quantile", subtitle = paste0(tool_icon(), " ISOFOX")) +
    base_theme + no_y_title + no_legend +
    theme(
      strip.text    = element_text(size = 8, face = "bold"),
      axis.title.x  = element_text(size = 9, color = "#555555"),
      axis.text.x   = element_text(size = 7),
      plot.subtitle = element_markdown(hjust = 0, size = 11, color = "#304b99",
                                       face = "bold", margin = margin(b = 4))
    )
} else NULL


# 5 - Statistical tests

In [ ]:
numeric_vars <- c(
  "tmbPerMb", "msIndelsPerMb", "snpPerMb", "indelsPerMb", "mnpPerMb",
  "purity", "ploidy", "chrArmDeviations", "lohGenomePct", "meanClonality",
  "hrd", "somaticGermlineTelomereRatio",
  "svDeletions", "svDuplications", "svInversions", "svTranslocations",
  "svFoldBack", "svComplexClusters", "svLine", "svDoubleMinute",
  "cdr3Total", "cdr3Tcell", "cdr3Bcell", "viralTotal"
)

tests_df <- bind_rows(lapply(numeric_vars, function(var) {
  tmp <- base %>% filter(!is.na(.data[[var]]))
  if (n_distinct(tmp$cohort) < 2) return(NULL)
  res <- wilcox.test(tmp[[var]] ~ tmp$cohort, exact = FALSE)
  data.frame(
    field   = var,
    p_value = res$p.value,
    higher  = median(tmp[[var]][tmp$cohort == plt_cohort], na.rm = TRUE) >
              median(tmp[[var]][tmp$cohort == ref_cohort],  na.rm = TRUE)
  ) %>%
  mutate(label = case_when(
    p_value < .01 &  higher ~ "▲",
    p_value < .01 & !higher ~ "▼",
    TRUE                    ~ ""
  ))
}))

# 4 - Driver burden

In [ ]:
# compute total drivers (oncogene + TSG) before plotting
base <- base %>% mutate(driverTotal = driverOnco + driverTsg)

driver_plot <- make_violin_panel(
  base,
  var_labels  = c(driverTotal = "Total drivers",
                  driverOnco  = "Oncogenes",
                  driverTsg   = "TSGs"),
  log_vars    = c("driverTotal", "driverOnco", "driverTsg"),
  fixed_y     = TRUE,
  panel_title = "Driver burden",
  tool_label  = "PURPLE · LINX · VirusInterpreter"
)

# 6 - Assemble & display

# 7 - Save

In [ ]:
# ── Unit test: make_violin_panel smoke test ───────────────────────────────────
local({
  dummy <- data.frame(
    sampleId = paste0("S", 1:20),
    cohort   = rep(c("A", "B"), 10),
    x        = c(runif(10, 0, 1), runif(10, 0.5, 1.5)),
    y        = c(rnorm(10, 5), rnorm(10, 8))
  )
  p <- make_violin_panel(dummy, var_labels = c(x = "Var X", y = "Var Y"),
                         panel_title = "Test", tool_label = "TEST")
  stopifnot(inherits(p, "gg"))
  message("\u2713 make_violin_panel: returns gg object")

  # pct_vars formats median label as %
  dummy2 <- data.frame(sampleId = paste0("S", 1:20),
                       cohort   = rep(c("A", "B"), 10),
                       purity   = runif(20, 0.1, 1))
  p2 <- make_violin_panel(dummy2, var_labels = c(purity = "Purity"),
                          pct_vars = "purity", panel_title = "", tool_label = "")
  stopifnot(inherits(p2, "gg"))
  message("\u2713 make_violin_panel: pct_vars accepted without error")
})

In [ ]:
figures <- list(
  tumor_char     = tumor_char_plot,
  genome         = genome_struct_plot,
  mut_burden     = mut_burden_plot,
  sv             = sv_plot,
  telomere       = telomere_plot,
  immune         = immune_plot,
  cuppa          = cuppa_plot,
  cuppa_extended = cuppa_extended_plot,
  cuppa_bar      = cuppa_bar_plot,
  cuppa_stacked  = cuppa_stacked_plot,
  driver         = driver_plot,
  rna            = rna_plot,
  rna_density    = rna_density_plot,
  rna_quantile   = rna_quantile_plot
)
saveRDS(figures, paste0(READY_FIG, "markers.rds"))